In [1]:
!nvidia-smi

Mon Jan 26 08:37:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
!pip install -q faiss-cpu sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 41.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [1]:
!pip install -U git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-esjrik_n
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-esjrik_n
  Resolved https://github.com/huggingface/transformers.git to commit be0115e42fbf5d7ffe915e01b0cf9a68b29958c6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 9.6 MB/s eta 0:00:00a 0:00:01
  Created wheel for transformers: filename=transformers-5.0.1.dev0-py3-none-any.whl size=11212445 sha256=b034ca7a3d5c92a16ea369f17b5e083954262c49af0d1f5e1949921357afbc7e
  Stored in directory: /tmp/pip-ephem-wheel-cache-ejqwv6c6/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggi

In [2]:
!pip install qwen-vl-utils accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 11.6 MB/s eta 0:00:0000:0100:01m


In [3]:
!pip install torch torchvision pillow opencv-python

In [4]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [7]:
from huggingface_hub import login
login(secret_value_0)

In [9]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    trust_remote_code=True,
    _from_auto=False
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Screenshot 2025-08-01 152934.png to Screenshot 2025-08-01 152934.png


In [5]:
image_path = list(uploaded.keys())[0]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "Describe this image in detail."}
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=150)

output = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print(output[0])

The image depicts a giant panda walking through a natural habitat. The panda is prominently featured in the center of the frame, occupying a significant portion of the image. It is walking on a grassy terrain with some patches of dry leaves and vegetation. The panda's fur is predominantly white with black patches on its face, ears, and paws. Its black ears are large and rounded, and its black eyes are prominent. The panda appears to be in a relaxed posture, with its front paws slightly raised, suggesting it might be drinking or sniffing the ground.

The background consists of a mix of grass, small plants, and some fallen leaves, indicating a natural, possibly forested area. The lighting suggests it is daytime, with sunlight filtering through


In [19]:
from google.colab import files
uploaded = files.upload()

Saving inputpic.jpg to inputpic.jpg


In [14]:
import cv2
import torch
from datetime import timedelta
from PIL import Image

In [10]:
image_path = '/kaggle/input/test-inputs/gun.jpg'

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "You are a vigilant observer tasked with documenting exactly what is happening in the current frame of video footage. Describe the scene as precisely and objectively as possible, focusing on visible actions, objects, people, positions, and interactions. Be concise and direct, but do not omit any details — even minor or seemingly irrelevant ones may be important later. Do not summarize across time or interpret intent. Just report what is visible in this single frame."}
        ],
    }
]

text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=150)

output = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print(output[0])

The image depicts a person wearing a black hooded sweatshirt and a black mask, holding a handgun in a defensive stance. The background is blurred, suggesting a focus on the subject. The individual appears to be in a darkened room or an enclosed space, with a metallic or reflective surface visible in the background. The scene conveys a sense of tension or danger.


In [11]:
def caption_frame(image, processor, model, device="cuda"):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "You are a vigilant observer tasked with documenting exactly what is happening "
                        "in the current frame of video footage. Describe the scene as precisely and "
                        "objectively as possible, focusing on visible actions, objects, people, positions, "
                        "and interactions. Be concise and direct, but do not omit any details. "
                        "Do not interpret intent or summarize across time."
                    )
                }
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=150)

    output = processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return output[0].strip()


In [12]:
def process_video_and_generate_report(
    video_path,
    processor,
    model,
    output_txt_path="video_captions.txt",
    interval_seconds=2,
    device="cuda"
):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError("Could not open video file")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * interval_seconds)

    captions_log = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            timestamp_seconds = frame_idx / fps
            timestamp = str(timedelta(seconds=int(timestamp_seconds)))

            # Convert BGR → RGB → PIL
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)

            caption = caption_frame(pil_image, processor, model, device)

            entry = f"[{timestamp}] {caption}"
            captions_log.append(entry)

            print(entry)  # optional live logging

        frame_idx += 1

    cap.release()

    # Write to text file
    full_text = "\n\n".join(captions_log)
    with open(output_txt_path, "w") as f:
        f.write(full_text)

    return output_txt_path


In [16]:
video_path = "/kaggle/input/test-inputs/input4.mp4"

output_file = process_video_and_generate_report(
    video_path=video_path,
    processor=processor,
    model=model,
    output_txt_path="/kaggle/working/video_analysis.txt",
    interval_seconds=1
)

print("Saved to:", output_file)

[0:00:00] The scene depicts an outdoor urban setting, likely a street or alleyway. A motor scooter is parked on the left side of the frame, with its rear light illuminated. The scooter is partially obscured by a person standing on the right side of the frame, holding a broom. The person appears to be in motion, possibly walking or moving towards the scooter. The background includes a bench and a wall, suggesting an urban environment. The overall lighting suggests it is daytime. The text overlay indicates that a CCTV footage of a man being shot at in Delhi's Shahdara area has surfaced online.
[0:00:01] The video frame depicts an outdoor scene in a street area. There is a man walking away from the camera, wearing a white shirt. In the foreground, there is a parked motorbike with its headlight on. The background shows a building with a shuttered window and a concrete sidewalk. The overall setting appears to be a quiet, urban environment.
[0:00:02] The scene depicts an outdoor urban settin